# Phase 2 Run 2 — MCD-rPPG Real-World Fine-Tuning

**Start:** `checkpoints/phase1/best.pth` (epoch 6, cache-composite 1.545)  
**Data:** MCDrPPGCachedDataset — FullHDwebcam frontal only, YOLO5Face 72×72 crop  
**Curriculum:** Soft SCAMPS phaseout: fraction = max(0, 1 − epoch/10). SCAMPS gone by epoch 10.  
**Effective batch:** 64 (8 per GPU × 2 GPUs × 4 accum steps), lr_max=3e-5  
**Exit gate:** Composite score improves over Phase 1 starting point (score < 1.545).  
**Output:** `checkpoints/phase2/best.pth`

Composite score formula (lower = better):  
`score = 0.3*(UBFC_MAE/1.23) + 0.3*(MCD_MAE/12.36) + 0.4*(rPPG10_MAE/13.89)`

Run 2 fixes vs Run 1:
- Training/eval input mismatch fixed (YOLO5Face crop in both)
- Frontal camera only (no USBVideo/IriunWebcam noise)
- Soft curriculum (no loss spike)
- lr_max=3e-5 (down from 1e-4)
- Gradient accumulation ×4
- Per-clip loss cap (max=1.5)
- SNR filter in eval
- ROI subregion augmentation


In [ ]:
import sys, os, json, math, warnings
from pathlib import Path
from collections import OrderedDict
from datetime import datetime

import numpy as np
import torch
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('/mnt/sata-ssd/rppg_project')
FP_ROOT      = PROJECT_ROOT / 'external' / 'FactorizePhys'
sys.path.insert(0, str(FP_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

OUT_DIR      = PROJECT_ROOT / 'checkpoints' / 'phase2'
CFG_PATH     = OUT_DIR / 'config.json'
METRICS_JSON = OUT_DIR / 'metrics.json'
BEST_CKPT    = OUT_DIR / 'best.pth'
TRAIN_SCRIPT = PROJECT_ROOT / 'src' / 'train_phase2.py'
PLOTS_DIR    = PROJECT_ROOT / 'docs' / 'plots'

# Eval caches (MCD .pt is built at training startup from held-out H5 files)
UBFC_CACHE   = PROJECT_ROOT / 'eval_caches' / 'ubfc_eval_cache.pt'
RPPG10_CACHE = PROJECT_ROOT / 'eval_caches' / 'rppg10_eval_cache.pt'
MCD_EVAL_DIR = PROJECT_ROOT / 'eval_caches' / 'mcd_eval_cache'
MCD_CACHE    = PROJECT_ROOT / 'eval_caches' / 'mcd_eval_cache.pt'

START_CKPT   = PROJECT_ROOT / 'checkpoints' / 'phase1' / 'best.pth'
MCD_SPLIT    = PROJECT_ROOT / 'checkpoints' / 'mcd_split.json'

WORLD_SIZE = torch.cuda.device_count()
print(f'CUDA devices:  {WORLD_SIZE}')
print(f'Start ckpt:    {START_CKPT} (exists={START_CKPT.exists()})')
print(f'Config:        {CFG_PATH} (exists={CFG_PATH.exists()})')
print(f'Train script:  {TRAIN_SCRIPT} (exists={TRAIN_SCRIPT.exists()})')
print(f'UBFC cache:    {UBFC_CACHE} (exists={UBFC_CACHE.exists()})')
print(f'rPPG10 cache:  {RPPG10_CACHE} (exists={RPPG10_CACHE.exists()})')
mcd_h5_count = len(list(MCD_EVAL_DIR.glob("*.h5"))) if MCD_EVAL_DIR.exists() else 0
print(f'MCD eval H5s:  {mcd_h5_count} files in {MCD_EVAL_DIR}')


In [ ]:
# ── ClearML init ──────────────────────────────────────────────────────────────
RUN_DATE = datetime.now().strftime('%Y%m%d_%H%M')
CLEARML_TASK_ID = None

try:
    from clearml import Task
    task = Task.init(
        project_name='rppg',
        task_name=f'phase2_run2/{RUN_DATE}',
        reuse_last_task_id=False,
    )
    CLEARML_TASK_ID = task.id
    task.set_parameters({
        'start_ckpt':           str(START_CKPT),
        'epochs':               20,
        'batch_size':           8,
        'grad_accum_steps':     4,
        'effective_batch':      64,
        'lr_max':               3e-5,
        'curriculum_end_epoch': 10,
        'dataset':              'MCD-rPPG FullHDwebcam frontal only, 500 train subjects',
        'fixes':                'YOLO5Face crop, soft curriculum, grad_accum×4, loss_cap, SNR filter',
    })
    print(f'ClearML task id: {CLEARML_TASK_ID}')
except Exception as e:
    print(f'ClearML unavailable: {e}')
    task = None

# Patch task ID into config
cfg = json.loads(CFG_PATH.read_text())
cfg['clearml_task_id'] = CLEARML_TASK_ID
CFG_PATH.write_text(json.dumps(cfg, indent=2))
print('Config updated with ClearML task id')


In [ ]:
# ── Verify eval caches ────────────────────────────────────────────────────────
for name, path in [('UBFC', UBFC_CACHE), ('rPPG-10', RPPG10_CACHE)]:
    if not path.exists():
        raise FileNotFoundError(f'{name} cache missing: {path}')
    clips = torch.load(str(path), weights_only=False)
    print(f'{name:8s}: {len(clips):4d} clips  ({path.stat().st_size/1e9:.2f} GB)  keys={list(clips[0].keys())}')

# MCD eval cache is built from H5 files at training startup — verify H5s are present
mcd_h5_files = sorted(MCD_EVAL_DIR.glob("*.h5")) if MCD_EVAL_DIR.exists() else []
if len(mcd_h5_files) == 0:
    raise FileNotFoundError(
        f'MCD eval H5 files missing: {MCD_EVAL_DIR}\n'
        f'Run: python src/cache/mcd_cache_builder.py --split held_out'
    )
print(f'MCD H5 files: {len(mcd_h5_files)} in {MCD_EVAL_DIR}')

print()
print(f'Start ckpt size: {START_CKPT.stat().st_size/1e6:.1f} MB')


In [ ]:
# ── Estimate training dataset size ────────────────────────────────────────────
from datasets.mcd_rppg_cached import MCDrPPGCachedDataset

mcd_ds_probe = MCDrPPGCachedDataset(
    split_json=str(MCD_SPLIT),
    split='train',
    steps=['before', 'after'],
    clip_len=160,
    stride=160,
    transform=None,
    seed=42,
)
print(f'MCD frontal train clips available: {len(mcd_ds_probe)}')
del mcd_ds_probe


In [ ]:
# ── Launch DDP training ───────────────────────────────────────────────────────
import subprocess, sys

assert TRAIN_SCRIPT.exists(), f'Missing: {TRAIN_SCRIPT}'
assert CFG_PATH.exists(),     f'Missing: {CFG_PATH}'
assert START_CKPT.exists(),   f'Missing: {START_CKPT}'

cmd = [
    'torchrun',
    f'--nproc_per_node={WORLD_SIZE}',
    '--rdzv_backend=c10d',
    '--rdzv_endpoint=localhost:29501',
    str(TRAIN_SCRIPT),
]
print('Launching:', ' '.join(cmd))
print()

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=str(PROJECT_ROOT),
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print(f'\nTraining finished — exit code: {proc.returncode}')
if proc.returncode != 0:
    raise RuntimeError(f'Training failed with exit code {proc.returncode}')

In [ ]:
# ── Final report ──────────────────────────────────────────────────────────────
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

with open(str(METRICS_JSON)) as f:
    hist = json.load(f)

best_row = min(hist, key=lambda r: r['score'])
print(f"Best epoch: {best_row['epoch']}")
print(f"  UBFC MAE  : {best_row['ubfc_mae']:.2f} bpm")
print(f"  MCD MAE   : {best_row['mcd_mae']:.2f} bpm")
print(f"  rPPG10 MAE: {best_row['rppg10_mae']:.2f} bpm")
print(f"  Score     : {best_row['score']:.4f}")
print(f"Best checkpoint: {BEST_CKPT}")

# Phase 1 reference (cache eval)
P1_UBFC, P1_MCD, P1_R10 = 1.70, 18.74, 23.47
P1_SCORE = 0.3*(P1_UBFC/1.23) + 0.3*(P1_MCD/12.36) + 0.4*(P1_R10/13.89)
print(f'\nPhase 1 reference: UBFC={P1_UBFC} MCD={P1_MCD} rPPG10={P1_R10} score={P1_SCORE:.4f}')
delta = best_row['score'] - P1_SCORE
print(f'Delta vs Phase 1: {delta:+.4f} ({"IMPROVED" if delta < 0 else "REGRESSED"})')

print('\n--- Per-Epoch Table ---')
print(f'{"Epoch":>6}  {"Loss":>8}  {"UBFC":>6}  {"MCD":>7}  {"rPPG10":>7}  {"Score":>8}  Best')
best_score_seen = float('inf')
for r in hist:
    mark = ''
    if r['score'] < best_score_seen:
        best_score_seen = r['score']
        mark = '★'
    print(f"{r['epoch']:>6}  {str(r.get('train_loss','N/A')):>8}  {r['ubfc_mae']:>6.2f}  "
          f"{r['mcd_mae']:>7.2f}  {r['rppg10_mae']:>7.2f}  {r['score']:>8.4f}  {mark}")

# ── Plots ─────────────────────────────────────────────────────────────────────
epochs   = [r['epoch']      for r in hist]
scores   = [r['score']      for r in hist]
ubfc_m   = [r['ubfc_mae']   for r in hist]
mcd_m    = [r['mcd_mae']    for r in hist]
r10_m    = [r['rppg10_mae'] for r in hist]
losses   = [r.get('train_loss') for r in hist]

CURRICULUM_END = 10  # soft phaseout ends at epoch 10

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Phase 2 Run 2 Training Curves', fontsize=13)

ax = axes[0]
ax.plot(epochs, scores, 'b-o', ms=4)
ax.axhline(P1_SCORE, color='orange', ls='--', label=f'Phase1 start ({P1_SCORE:.3f})')
ax.axvline(CURRICULUM_END, color='gray', ls=':', alpha=0.6, label=f'Curriculum end (ep {CURRICULUM_END})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Composite Score')
ax.set_title('Composite Score (lower=better)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(epochs, ubfc_m, 'g-o', ms=4, label='UBFC')
ax.plot(epochs, mcd_m,  'r-o', ms=4, label='MCD')
ax.plot(epochs, r10_m,  'b-o', ms=4, label='rPPG-10')
ax.axhline(P1_UBFC, color='g', ls='--', alpha=0.5)
ax.axhline(P1_MCD,  color='r', ls='--', alpha=0.5)
ax.axhline(P1_R10,  color='b', ls='--', alpha=0.5)
ax.axvline(CURRICULUM_END, color='gray', ls=':', alpha=0.6)
ax.set_xlabel('Epoch'); ax.set_ylabel('MAE (bpm)')
ax.set_title('Per-Dataset MAE')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[2]
valid_losses = [(e, l) for e, l in zip(epochs, losses) if l is not None]
if valid_losses:
    ep_l, lo_l = zip(*valid_losses)
    ax.plot(ep_l, lo_l, 'k-o', ms=4)
ax.axvline(CURRICULUM_END, color='gray', ls=':', alpha=0.6, label=f'Curriculum end (ep {CURRICULUM_END})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Train Loss')
ax.set_title('Training Loss')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = PLOTS_DIR / 'phase2_training_curves.png'
plt.savefig(str(plot_path), dpi=120, bbox_inches='tight')
plt.show()
print(f'Plot saved: {plot_path}')


In [ ]:
# ── Update docs ───────────────────────────────────────────────────────────────
from datetime import date
import re

with open(str(METRICS_JSON)) as f:
    hist = json.load(f)

best_row = min(hist, key=lambda r: r['score'])
today = date.today().isoformat()

exec_entry = (
    f"## {today} — Phase 2 Run 2 — MCD-rPPG Frontal Fine-Tuning\n\n"
    f"**Notebook: `notebooks/training/phase2_multidataset_finetune.ipynb`**\n"
    f"- Start from `checkpoints/phase1/best.pth` (epoch 6, cache-composite 1.545)\n"
    f"- MCD frontal only (FullHDwebcam, YOLO5Face crop, 72×72)\n"
    f"- Soft SCAMPS curriculum phaseout over 10 epochs\n"
    f"- 2×RTX3060 DDP, effective batch 64 (8×2×4 accum), 20 epochs, lr_max=3e-5\n"
    f"- Loss: NegPearson (cap=1.5) + 0.2×freq, float32 TF32\n"
    f"- SNR filter in eval (threshold=2.0), ROI subregion augmentation\n"
    f"- Best checkpoint: epoch {best_row['epoch']} (score {best_row['score']:.4f})\n"
    f"- Best: UBFC={best_row['ubfc_mae']:.2f} MCD={best_row['mcd_mae']:.2f} rPPG10={best_row['rppg10_mae']:.2f} bpm\n"
    f"- ClearML task id: {CLEARML_TASK_ID}\n\n"
)

exec_log_path = PROJECT_ROOT / 'docs' / 'execution_log.md'
existing = exec_log_path.read_text()
exec_log_path.write_text(exec_entry + existing)
print('execution_log.md updated')

# Update progress.md Phase 2 block
progress_path = PROJECT_ROOT / 'docs' / 'progress.md'
progress_text = progress_path.read_text()

p2_block = (
    f"4. **[COMPLETE] Phase 2 — MCD-rPPG real-world fine-tuning (Run 2)**\n"
    f"   - Start: `checkpoints/phase1/best.pth` (epoch 6, cache-composite 1.545)\n"
    f"   - Best epoch: {best_row['epoch']} | score={best_row['score']:.4f} | "
    f"UBFC={best_row['ubfc_mae']:.2f} MCD={best_row['mcd_mae']:.2f} rPPG10={best_row['rppg10_mae']:.2f}\n"
    f"   - Checkpoint: `checkpoints/phase2/best.pth`\n"
    f"   - Last updated: {today}"
)

updated = re.sub(
    r'4\. \*\*\[IN PROGRESS\] Phase 2.*?(?=\n\n\d+\.|\Z)',
    p2_block,
    progress_text,
    flags=re.DOTALL,
)
if updated != progress_text:
    progress_path.write_text(updated)
    print('progress.md updated')
else:
    print('progress.md Phase 2 section not matched — check regex')
